# zero-grad-set-none — worked example 1: zero_grad via set_to_none

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `zero-grad-set-none`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The modern `zero_grad` convention sets each parameter's `.grad` attribute to `None` rather than zeroing an existing buffer. `None` is treated as zero on the next access and saves the memory of a zero tensor. The next `.backward()` allocates a fresh `.grad` automatically.

## Worked solution

We implement a hand-rolled optimizer's `zero_grad`.

1. We iterate the parameter list. Each `p` is a leaf tensor with `requires_grad=True`.
2. We set `p.grad = None`. We deliberately do NOT call `.zero_()` or allocate `zeros_like` — the None convention is the point and it is cheaper.
3. After the call, every parameter has `p.grad is None`; PyTorch will re-allocate the gradient on the next backward.

We populate fake gradients, run `zero_grad`, and print that every grad is now `None`.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(0)
model = nn.Linear(4, 2)
params = list(model.parameters())
# fake a backward by assigning grads
for p in params:
    p.grad = t.ones_like(p)

def zero_grad(params):
    for p in params:
        p.grad = None

print('grads before:', [p.grad is not None for p in params])
zero_grad(params)
print('all None after:', all(p.grad is None for p in params))